# How to Solve a QUBO / Ising Problem

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/solve_qubo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fsolve_qubo.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/solve_qubo.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


QuPrep's `qubo` module provides problem formulations (Max-Cut, Knapsack), brute-force and simulated annealing solvers, QUBO ↔ Ising conversion, and QAOA circuit export.

In [ ]:
!pip install -q quprep

In [ ]:
import numpy as np

import quprep as qd
from quprep.qubo import ising_to_qubo, knapsack, max_cut, qaoa_circuit, qubo_to_ising
from quprep.qubo.solver import solve_brute, solve_sa

print(f"quprep {qd.__version__}")

## 1. Max-Cut formulation

`max_cut()` takes a weighted adjacency matrix. Here we build a 4-cycle graph.

In [ ]:
adj = np.array(
    [[0, 1, 0, 1],
     [1, 0, 1, 0],
     [0, 1, 0, 1],
     [1, 0, 1, 0]],
    dtype=float,
)
q_maxcut = max_cut(adj)
print(f"QUBO matrix shape : {q_maxcut.Q.shape}")
print(f"Offset            : {q_maxcut.offset}")

## 2. Brute-force solver

In [ ]:
sol_brute = solve_brute(q_maxcut)
print(f"Best assignment : {sol_brute.x.astype(int).tolist()}")
print(f"Objective value : {sol_brute.energy:.4f}")

## 3. Simulated annealing

In [ ]:
sol_sa = solve_sa(q_maxcut, n_steps=500, seed=42)
print(f"Best assignment : {sol_sa.x.astype(int).tolist()}")
print(f"Objective value : {sol_sa.energy:.4f}")

## 4. Knapsack problem

In [ ]:
weights  = np.array([2, 3, 4, 5], dtype=float)
values   = np.array([3, 4, 5, 6], dtype=float)
capacity = 8
q_knap = knapsack(weights, values, capacity=capacity)
sol_knap = solve_brute(q_knap)
print(f"Values/weights : {list(zip(values.astype(int).tolist(), weights.astype(int).tolist()))}")
print(f"Capacity       : {capacity}")
print(f"Best items     : {sol_knap.x.astype(int).tolist()}")
print(f"Objective      : {sol_knap.energy:.4f}")

## 5. QUBO ↔ Ising round-trip

In [ ]:
ising = qubo_to_ising(q_maxcut)
print(f"h (linear)    : {ising.h}")
print(f"J (quadratic) :\n{ising.J}")
print(f"offset        : {ising.offset:.4f}")
q_back = ising_to_qubo(ising)
print(f"Round-trip OK : {np.allclose(q_maxcut.Q, q_back.Q)}")

## 6. QAOA circuit export

`qaoa_circuit()` returns an OpenQASM 3.0 string ready for any QASM-compatible backend.

In [ ]:
qasm_str = qaoa_circuit(q_maxcut, p=1)
lines = qasm_str.strip().splitlines()
print(f"QASM lines : {len(lines)}")
print("\n".join(lines[:10]), "...")